# Dorado: Two-Stage Reasoning Post-Training

**First time?** Run `bash setup.sh` in a terminal first, then restart kernel.

In [1]:
import os, sys, subprocess
from pathlib import Path

os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"

# Runtime cache
repo_candidates = ["/home/jovyan/llmpt", os.getcwd()]
repo_root = next((p for p in repo_candidates if os.path.isdir(p)), os.getcwd())
cache_dir = os.path.join(repo_root, ".runtime_cache")
Path(cache_dir).mkdir(parents=True, exist_ok=True)
for key, sub in [("DORADO_RUNTIME_CACHE", ""), ("HF_HOME", "huggingface"),
                  ("HF_HUB_CACHE", "huggingface/hub"), ("HF_DATASETS_CACHE", "huggingface/datasets"),
                  ("TRANSFORMERS_CACHE", "huggingface/transformers"), ("PIP_CACHE_DIR", "pip")]:
    os.environ[key] = os.path.join(cache_dir, sub) if sub else cache_dir

# GPU
try:
    rows = subprocess.check_output(
        ["nvidia-smi", "--query-gpu=index,memory.free", "--format=csv,noheader,nounits"],
        text=True).strip().splitlines()
    selected = [r.split(",")[0].strip() for r in sorted(rows, key=lambda r: int(r.split(",")[1]), reverse=True)[:8]]
except Exception:
    selected = ["0"]
os.environ["CUDA_VISIBLE_DEVICES"] = ",".join(selected)

# Path
for p in [os.path.abspath("."), os.path.expanduser("~/llmpt")]:
    if os.path.isdir(os.path.join(p, "dorado")):
        if p not in sys.path: sys.path.insert(0, p)
        os.chdir(p)
        break

import torch, transformers
print(f"torch={torch.__version__}  transformers={transformers.__version__}  GPU={os.environ['CUDA_VISIBLE_DEVICES']}")
if torch.cuda.is_available():
    print(f"🎯 {torch.cuda.get_device_name(0)} ({torch.cuda.device_count()} visible)")

from dorado import (get_profile, PROFILES, build_experiment_grid, make_results_paths,
    run_sft_stage, run_candidate_generation, run_labeling_stage,
    run_dpo_training, run_full_evaluation,
    run_single_experiment, run_all_experiments, cleanup_artifacts,
    clear_gpu, set_random_seeds, cleanup_storage)
print(f"✅ dorado loaded")

torch=2.4.1+cu121  transformers=4.49.0  GPU=0
🎯 NVIDIA GeForce RTX 3090 (1 visible)
✅ dorado loaded


/opt/conda/lib/python3.11/site-packages/transformers/utils/hub.py:106: FutureWarning: Using `TRANSFORMERS_CACHE` is deprecated and will be removed in v5 of Transformers. Use `HF_HOME` instead.
  warnings.warn(


In [2]:
PROFILE = "smoke"  # "smoke" | "fast" | "full"
OVERRIDES = None

EXPERIMENTS = build_experiment_grid(profile=PROFILE, overrides=OVERRIDES)
_, RESULTS_FILE, CHECKPOINT_FILE = make_results_paths()
cfg = EXPERIMENTS[0]
print(f"{PROFILE}: {cfg['base_model']} | SFT={cfg['sft_samples']} | Math={cfg['math_prompt_count']} | β={cfg['dpo_beta']}")

smoke: HuggingFaceTB/SmolLM2-135M | SFT=50 | Math=20 | β=0.1


In [ ]:
results_df = run_all_experiments(EXPERIMENTS, RESULTS_FILE, CHECKPOINT_FILE)

No checkpoint found. Starting fresh.

🧹 Pre-run cleanup…
🧹 Cleanup found no intermediate artifacts to remove

RUNNING 1 EXPERIMENTS


EXPERIMENT 1/1 (ID: 0)

EXPERIMENT 0 (profile=smoke)
🧹 Clearing intermediate artifacts before experiment run…
🧹 Cleanup found no intermediate artifacts to remove

🎮 GPU Memory: 0.00 GB allocated, 0.00 GB reserved
🎲 Random seeds set to 42

[Stage 1] Cold-Start SFT (GC-Boost)…
trainable params: 921,600 || all params: 135,436,608 || trainable%: 0.6805
📦 SFT dataset: HuggingFaceH4/ultrachat_200k (50 samples, streamed)


Tokenizing SFT data (num_proc=8):   0%|          | 0/50 [00:00<?, ? examples/s]

No label_names provided for model class `PeftModelForCausalLM`. Since `PeftModel` hides base models input arguments, if label_names is not given, label_names can't be set automatically within `Trainer`. Note that empty label_names list will be used instead.
`use_cache=True` is incompatible with gradient checkpointing. Setting `use_cache=False`.
/opt/conda/lib/python3.11/site-packages/torch/utils/checkpoint.py:1399: FutureWarning: `torch.cpu.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cpu', args...)` instead.
  with device_autocast_ctx, torch.cpu.amp.autocast(**cpu_autocast_kwargs), recompute_context:  # type: ignore[attr-defined]


Step,Training Loss
10,2.263300
20,2.049800


✅ SFT (GC-Boost) complete. Saved to 'coldstart_dorado'

DPO ROUND 1/1

[Stage 2] Candidate Generation (Round 1)…
⚠️  [WARNING] MATH data not found at /home/jovyan/llmpt/eval/data/math/test.jsonl. Attempting to load from HuggingFace datasets...
📦 Loaded 20 MATH prompts from HuggingFace


Generating Candidates:   0%|          | 0/20 [00:00<?, ?it/s]

In [ ]:
if results_df is not None and not results_df.empty:
    display(results_df)
else:
    print("No results yet.")
cleanup_artifacts()